this is for assigment 1

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
pd.set_option('display.max_rows', 100)

In [3]:
tickers = pd.read_csv('data_export/data_us/tickers.csv',names=['ticker','GISC code'])
mkt_cap = pd.read_csv('data_export/data_us/mktcap.csv')
sp500 = pd.read_csv('data_export/data_us/univ_h.csv')

In [5]:
# tickers 数据处理，添加 sectors 和 industries 列
tickers['GISC sectors'] = tickers['GISC code'].str[:2]
tickers['GISC industries'] = tickers['GISC code'].str[:6]
tickers.head(10)

,ticker,GISC code,GISC sectors,GISC industries
0,0111145D,NaN,NaN,NaN
1,0202445Q,NaN,NaN,NaN
2,0203524D,NaN,NaN,NaN
3,0226226D,NaN,NaN,NaN
4,0544749D,NaN,NaN,NaN
5,0574018D,NaN,NaN,NaN
6,0772031D,NaN,NaN,NaN
7,0848680D,NaN,NaN,NaN
8,0867887D,NaN,NaN,NaN
9,0910150D,25301020,25,253010


In [7]:
# market cap 2008、2025与 tickers 合并
mkt_cap_2008 = mkt_cap[mkt_cap['Date'].astype(str).str[:4]=='2008'].drop(columns=['Date']).iloc[0].reset_index(name='mkt_cap_2008')
mkt_cap_2008.rename(columns={'index':'ticker'}, inplace=True)
df_merged = pd.merge(tickers, mkt_cap_2008, on='ticker', how='right')
mkt_cap_2025 = mkt_cap[mkt_cap['Date'].astype(str).str[:4]=='2025'].drop(columns=['Date']).iloc[0].reset_index(name='mkt_cap_2025')
mkt_cap_2025.rename(columns={'index':'ticker'}, inplace=True)
df_merged = pd.merge(df_merged, mkt_cap_2025, on='ticker',how='left')
df_merged

,ticker,GISC code,GISC sectors,GISC industries,mkt_cap_2008,mkt_cap_2025
0,0111145D,NaN,NaN,NaN,1877.1800,NaN
1,0202445Q,NaN,NaN,NaN,3976.9300,NaN
2,0203524D,NaN,NaN,NaN,5652.3922,NaN
3,0226226D,NaN,NaN,NaN,2960.0640,NaN
4,0544749D,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
959,YUM,25301040,25,253010,19266.0595,37272.8368
960,ZBH,35101010,35,351010,15559.9470,20795.2400
961,ZBRA,45203010,45,452030,2330.2728,19794.3515
962,ZION,40101015,40,401010,4845.1015,7986.7456


注意google出现了两次！！

In [8]:
df_merged[df_merged['ticker'].isin(['GOOG','GOOGL'])]

,ticker,GISC code,GISC sectors,GISC industries,mkt_cap_2008,mkt_cap_2025
471,GOOG,50203010,50,502030,214355.1769,2325971.83
472,GOOGL,50203010,50,502030,214355.1769,2325971.83


In [23]:
#提取2008年和2025年的sp500成分股
row2008 = sp500[sp500['year']==2008].drop(columns="year").iloc[0]
row2025 = sp500[sp500['year']==2025].drop(columns="year").iloc[0]
tickers_2008 = row2008[row2008 == 1].index.tolist()
tickers_2025 = row2025[row2025 == 1].index.tolist()

In [13]:
print(row2008['GOOG'])
print(row2008['GOOGL'])
print(row2025['GOOG'])
print(row2025['GOOGL'])

0
1
1
1


In [25]:
# 2008 sector and industries
ticker2008 = df_merged[df_merged['ticker'].isin(tickers_2008)].copy()
ticker2008.dropna(subset=['GISC code'], inplace=True)
# ticker2008.dropna(subset=['mkt_cap_2008'], inplace=True)

# 2008 sector
sec2008 = ticker2008.groupby('GISC sectors').agg(number=('GISC sectors','size'),mktcap=('mkt_cap_2008','sum'))
sec2008.sort_values(by='mktcap', ascending=False)

,number,mktcap
GISC sectors,,
40,70,1.910081e+06
45,48,1.737036e+06
10,29,1.572799e+06
35,44,1.458662e+06
20,53,1.436799e+06
30,34,1.337934e+06
50,23,1.011713e+06
25,54,6.498488e+05
55,25,4.061667e+05


In [26]:
# 2008 industries
ind2008 = ticker2008.groupby('GISC industries').agg(number=('GISC industries','size'),mktcap=('mkt_cap_2008','sum'))
ind2008.sort_values(by='mktcap', ascending=False)

,number,mktcap
GISC industries,,
101020,19,1.259032e+06
401010,18,7.506683e+05
201010,12,7.130312e+05
352020,9,6.143383e+05
403010,23,5.389853e+05
451030,10,5.312700e+05
402030,19,4.407964e+05
501010,5,3.881952e+05
453010,16,3.879756e+05


In [27]:
# 2025 sector and industries
ticker2025 = df_merged[df_merged['ticker'].isin(tickers_2025)].copy()
ticker2025.dropna(subset=['GISC code'], inplace=True)

# 2025 sector
sec2025 = ticker2025.groupby('GISC sectors').agg(number=('GISC sectors','size'),mktcap=('mkt_cap_2025','sum'))
sec2025.sort_values(by='mktcap', ascending=False)

,number,mktcap
GISC sectors,,
45,69,1.646957e+07
50,22,7.777047e+06
40,73,7.443647e+06
25,50,6.072731e+06
35,61,5.227583e+06
20,78,4.187276e+06
30,38,3.201579e+06
10,22,1.658627e+06
55,31,1.171475e+06


In [29]:
grouped = ticker2025.groupby('GISC sectors')
for sector, group in grouped:
    if sector == '50':
        print(f"Sector: {sector}")
        print(group[['ticker', 'mkt_cap_2025']])
        print("\n")
        print(f"Sum of market cap for sector {sector}: {group['mkt_cap_2025'].sum()}")

Sector: 50
    ticker  mkt_cap_2025
278   CHTR  5.557826e+04
288  CMCSA  1.431891e+05
352    DIS  2.006883e+05
376     EA  3.826560e+04
438    FOX  2.172766e+04
439   FOXA  2.172766e+04
471   GOOG  2.325972e+06
472  GOOGL  2.325972e+06
525    IPG  1.047867e+04
600    LYV  3.002707e+04
620   META  1.513317e+06
647   MTCH  8.188077e+03
670   NFLX  3.790399e+05
688    NWS  1.616811e+04
689   NWSA  1.616811e+04
698    OMC  1.686385e+04
706   PARA  7.531191e+03
835      T  1.638119e+05
858   TMUS  2.547616e+05
875   TTWO  3.215211e+04
919     VZ  1.692691e+05
924    WBD  2.615074e+04


Sum of market cap for sector 50: 7777047.217100001


In [10]:
# 2025 industries
ind2025 = ticker2025.groupby('GISC industries').agg(number=('GISC industries','size'),mktcap=('mkt_cap_2025','sum'))
ind2025.sort_values(by='mktcap', ascending=False)

,number,mktcap
GISC industries,,
502030,4,6.173448e+06
453010,19,5.815256e+06
451030,21,5.422167e+06
452020,8,3.907135e+06
402010,10,2.486720e+06
255030,2,2.345443e+06
402030,23,1.808837e+06
401010,13,1.729980e+06
352020,7,1.690411e+06


In [11]:
#Q2
ticker2025[ticker2025['ticker']=='MCD']

,ticker,GISC code,GISC sectors,GISC industries,mkt_cap_2008,mkt_cap_2025
609,MCD,25301040,25,253010,68718.2147,209618.4244


In [12]:
stk_list = ticker2025[ticker2025['GISC industries']=='253010'].loc[:,'ticker'].tolist()

In [13]:
#Q3
adj_price = pd.read_csv('data_export/data_us/adjusted.csv')
#日期格式
adj_price["Date"] = pd.to_datetime(adj_price["Date"].astype(str), format="%Y%m%d")
adj_price = adj_price.sort_values("Date")
adj_price.head()

,Date,0111145D,0202445Q,0203524D,0226226D,0544749D,0574018D,0772031D,0848680D,0867887D,...,XOM,XRAY,XRX,XTO,XYL,YUM,ZBH,ZBRA,ZION,ZTS
0,2004-01-02,23.60,42.99,17.6344,46.1555,21.8167,23.1976,3.3329,14.1172,NaN,...,19.6993,18.1035,18.4457,12.3424,NaN,8.1331,60.2797,43.5867,39.2633,NaN
1,2004-01-05,23.72,43.06,18.2824,45.3553,22.2326,23.2143,3.5783,14.6964,NaN,...,20.1599,17.8036,18.7578,12.4431,NaN,8.2519,59.7372,43.6667,39.1785,NaN
2,2004-01-06,23.76,42.50,18.6019,46.5534,22.7548,23.1976,3.5625,15.2842,NaN,...,20.0242,17.8320,19.0972,12.2067,NaN,8.4993,59.4444,43.7667,39.7198,NaN
3,2004-01-07,23.52,43.39,18.3235,46.3323,22.7224,22.9974,3.5150,15.0854,NaN,...,19.8787,17.7388,19.1650,12.1498,NaN,8.3538,60.1074,44.5000,39.4459,NaN
4,2004-01-08,23.36,43.59,19.1678,43.9361,22.6808,22.9807,3.5783,15.3015,NaN,...,19.8302,17.8522,19.2193,12.2330,NaN,8.3781,61.2700,44.6000,39.6546,NaN


In [14]:
# 取出date 和 stk_list
adj_price_2125 = adj_price[adj_price['Date'].astype(str).str[:4].isin(['2020','2021','2022','2023','2024','2025'])].copy()
columes = ['Date'] + stk_list
adj_price_2125 = adj_price_2125.loc[:,columes].copy()
adj_price_2125.head()

,Date,ABNB,BKNG,CCL,CMG,CZR,DPZ,DRI,EXPE,HLT,LVS,MAR,MCD,MGM,NCLH,RCL,SBUX,WYNN,YUM
4027,2020-01-02,NaN,2041.180,50.7180,17.164,59.51,273.091,94.1440,109.684,109.9014,66.2794,145.5135,174.765,33.3851,58.83,131.0241,78.2219,138.4164,91.3225
4028,2020-01-03,NaN,2032.226,49.3144,17.303,58.01,274.273,94.2447,107.855,108.4617,65.2684,143.3811,174.147,33.0181,57.60,129.8953,77.7666,136.3633,91.0364
4029,2020-01-06,NaN,2014.437,47.8218,17.160,58.80,273.603,94.8652,107.470,107.8996,65.6147,141.5272,176.105,32.6214,56.80,128.0854,77.1538,136.0934,90.9828
4030,2020-01-07,NaN,2034.755,47.9701,17.202,59.33,271.649,94.0518,108.735,106.7853,65.6803,139.2219,176.367,32.6412,56.97,126.9275,76.9175,136.7392,91.1437
4031,2020-01-08,NaN,2029.688,48.1975,17.135,59.53,270.448,95.4774,107.351,107.6925,66.2794,140.9413,179.221,32.9685,57.57,128.6595,77.8104,137.5971,91.3046


In [15]:
#算log return
log_ret = (adj_price_2125.set_index("Date").apply(lambda x: np.log(x) - np.log(x.shift(1))))
log_ret.reset_index(inplace=True)
log_ret = log_ret[(log_ret['Date'].dt.year>=2021) & (log_ret['Date'].dt.year<=2025)].copy()

log_ret.head()

,Date,ABNB,BKNG,CCL,CMG,CZR,DPZ,DRI,EXPE,HLT,LVS,MAR,MCD,MGM,NCLH,RCL,SBUX,WYNN,YUM
253,2021-01-04,-0.053519,-0.028781,-0.060913,-0.049977,-0.036612,-0.010381,-0.024475,-0.007812,-0.034752,-0.028076,-0.055723,-0.020525,-0.059156,-0.069190,-0.054619,-0.036942,-0.053988,-0.025563
254,2021-01-05,0.063685,0.011024,0.010737,0.026926,0.043322,0.004837,0.015875,0.042986,0.004549,0.025387,0.010524,0.005976,0.021977,0.021264,0.025548,0.003292,0.030312,0.000094
255,2021-01-06,-0.038002,0.027826,-0.009756,-0.008672,0.019470,0.009216,0.023431,0.044777,0.025788,-0.015256,0.025835,-0.002276,0.026652,0.003706,-0.004835,0.007224,0.005973,0.005466
256,2021-01-07,0.057831,-0.004437,0.016529,0.018256,0.023592,-0.004375,0.027979,-0.007910,0.022495,-0.006685,0.007392,0.004638,-0.018777,0.013472,0.024354,-0.008094,-0.009973,-0.007642
257,2021-01-08,-0.009966,0.018442,-0.013104,0.024761,0.009435,0.014612,0.003222,0.012084,0.007650,-0.003963,-0.005076,0.018183,0.024533,-0.009371,-0.013746,0.022199,-0.006583,0.014571


In [16]:
log_ret[log_ret.isnull().any(axis=1)] #查看缺失值

,Date,ABNB,BKNG,CCL,CMG,CZR,DPZ,DRI,EXPE,HLT,LVS,MAR,MCD,MGM,NCLH,RCL,SBUX,WYNN,YUM


In [17]:
#行业平均
log_ret['industry simple average return'] = log_ret.mean(axis=1,numeric_only=True)
log_ret

,Date,ABNB,BKNG,CCL,CMG,CZR,DPZ,DRI,EXPE,HLT,LVS,MAR,MCD,MGM,NCLH,RCL,SBUX,WYNN,YUM,industry simple average return
253,2021-01-04,-0.053519,-0.028781,-0.060913,-0.049977,-0.036612,-0.010381,-0.024475,-0.007812,-0.034752,-0.028076,-0.055723,-0.020525,-0.059156,-0.069190,-0.054619,-0.036942,-0.053988,-0.025563,-0.039500
254,2021-01-05,0.063685,0.011024,0.010737,0.026926,0.043322,0.004837,0.015875,0.042986,0.004549,0.025387,0.010524,0.005976,0.021977,0.021264,0.025548,0.003292,0.030312,0.000094,0.020462
255,2021-01-06,-0.038002,0.027826,-0.009756,-0.008672,0.019470,0.009216,0.023431,0.044777,0.025788,-0.015256,0.025835,-0.002276,0.026652,0.003706,-0.004835,0.007224,0.005973,0.005466,0.008142
256,2021-01-07,0.057831,-0.004437,0.016529,0.018256,0.023592,-0.004375,0.027979,-0.007910,0.022495,-0.006685,0.007392,0.004638,-0.018777,0.013472,0.024354,-0.008094,-0.009973,-0.007642,0.008258
257,2021-01-08,-0.009966,0.018442,-0.013104,0.024761,0.009435,0.014612,0.003222,0.012084,0.007650,-0.003963,-0.005076,0.018183,0.024533,-0.009371,-0.013746,0.022199,-0.006583,0.014571,0.005994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1503,2025-12-24,0.002122,0.001330,-0.013035,0.002907,0.002452,-0.001761,0.015441,-0.005776,0.006568,0.001660,0.003465,0.007979,0.003241,0.002593,0.006241,0.008431,0.000480,-0.001360,0.002388
1504,2025-12-26,0.000292,-0.001170,-0.017757,-0.002377,0.011767,0.000306,-0.008142,0.003449,0.000614,-0.001660,0.001332,-0.008494,0.015782,-0.015659,-0.025745,0.006012,-0.005293,-0.007023,-0.002987
1505,2025-12-29,-0.001463,0.000219,0.000651,-0.017879,-0.031137,-0.010179,-0.007780,-0.002577,-0.003824,-0.008495,-0.003714,-0.006944,-0.013897,-0.017692,-0.010486,0.005743,-0.016949,-0.006350,-0.008486
1506,2025-12-30,0.002120,-0.002609,0.003250,-0.002696,-0.021452,0.001589,-0.010542,-0.006366,-0.006623,-0.008261,-0.001496,-0.001622,-0.011639,0.000892,-0.003508,-0.003747,-0.015994,-0.000591,-0.004961


In [18]:
mcd_ret = log_ret['MCD']
ind_ret = log_ret['industry simple average return']

df_reg = pd.concat([mcd_ret,ind_ret],axis=1)
df_reg.columns=['mcd','industry']

X = sm.add_constant(df_reg['industry'])
y = df_reg['mcd']

model = sm.OLS(y,X).fit()

alpha = model.params['const']
beta = model.params['industry']
r2 = model.rsquared


print("alpha:", alpha)
print("beta:", beta)
print("R-squared:", r2)


alpha: 0.00030691971960633275
beta: 0.24817097214099543
R-squared: 0.1481940635539113


In [19]:
#Q4
mkt_cap["Date"] = pd.to_datetime(mkt_cap["Date"].astype(str), format="%Y%m%d")

In [20]:
#日期
mkt_cap_2125 = mkt_cap[(mkt_cap['Date'].dt.year >=2021) & (mkt_cap['Date'].dt.year <=2025)].copy()
mkt_cap_2125 = mkt_cap_2125.set_index("Date").sort_index()
mkt_cap_2125.reset_index(inplace=True)
#stk list
columes = ['Date'] + stk_list
mkt_cap_2125 = mkt_cap_2125.loc[:,columes].copy()
mkt_cap_2125.set_index('Date',inplace=True)

mkt_cap_2125

,ABNB,BKNG,CCL,CMG,CZR,DPZ,DRI,EXPE,HLT,LVS,MAR,MCD,MGM,NCLH,RCL,SBUX,WYNN,YUM
Date,,,,,,,,,,,,,,,,,,
2021-01-04,83830.6741,88629.7060,21663.7070,36909.4380,14912.6431,14952.2643,15134.4404,18583.7881,29814.3804,44263.8400,40466.8682,156637.2319,14669.9912,7489.6496,15865.2766,121008.470,11531.3674,31922.5218
2021-01-05,89343.0756,89612.2143,21908.4046,37916.4507,15572.8816,15024.7602,15376.6123,19400.0205,29950.3291,45401.9439,40894.9861,157576.0717,14995.9910,7650.6155,16275.8176,121407.528,11886.2616,31925.5385
2021-01-06,86011.5368,92140.7706,21796.9160,37589.0807,15879.0490,15163.8418,15756.6754,20288.3983,30732.7276,44714.4986,41965.2807,157218.4185,15424.1536,7679.0213,16197.2988,122287.803,11957.4562,32100.5060
2021-01-07,91132.3469,91732.8596,22110.6962,38281.0347,16258.1134,15097.6500,16154.1763,20128.5469,31431.8923,44416.6056,42276.6392,157948.6272,15137.2386,7783.1757,16596.6228,121301.895,11838.7986,31856.1548
2021-01-08,90228.6746,93440.2699,21850.3891,39240.7607,16412.2385,15319.8654,16206.3076,20373.2751,31673.2706,44240.9251,42062.5802,160847.1090,15513.2031,7710.5832,16370.0401,124024.879,11761.1317,32323.7405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,84194.8956,175561.7957,41031.6540,50114.3362,5000.6446,14373.6707,22089.1465,35114.0133,68173.2342,44834.4778,84573.7517,223139.3225,10144.3539,10548.3160,80210.3393,96164.547,12998.8894,42847.3846
2025-12-26,84219.5176,175356.4663,40389.5065,49995.3312,5059.8359,14378.0629,21910.0233,35235.3200,68215.0725,44760.1030,84686.4595,221252.1135,10305.7227,10384.4233,77905.9147,96744.468,12930.2662,42547.5195
2025-12-29,84096.4077,175394.8246,40255.6500,49109.4049,4904.7139,14232.4443,21740.2053,35144.6463,67954.7451,44381.4677,84372.4879,219720.9816,10163.4993,10202.3203,77093.2301,97301.647,12712.9595,42278.1963


In [21]:
weights = mkt_cap_2125.div(mkt_cap_2125.sum(axis=1),axis=0)
weights

,ABNB,BKNG,CCL,CMG,CZR,DPZ,DRI,EXPE,HLT,LVS,MAR,MCD,MGM,NCLH,RCL,SBUX,WYNN,YUM
Date,,,,,,,,,,,,,,,,,,
2021-01-04,0.109114,0.115360,0.028197,0.048041,0.019410,0.019462,0.019699,0.024189,0.038806,0.057614,0.052672,0.203879,0.019094,0.009749,0.020650,0.157504,0.015009,0.041550
2021-01-05,0.114232,0.114576,0.028012,0.048479,0.019911,0.019210,0.019660,0.024804,0.038294,0.058050,0.052287,0.201473,0.019174,0.009782,0.020810,0.155229,0.015198,0.040819
2021-01-06,0.109582,0.117391,0.027770,0.047890,0.020231,0.019319,0.020075,0.025848,0.039155,0.056968,0.053466,0.200303,0.019651,0.009783,0.020636,0.155800,0.015234,0.040897
2021-01-07,0.115141,0.115900,0.027936,0.048366,0.020541,0.019075,0.020410,0.025431,0.039713,0.056118,0.053414,0.199560,0.019125,0.009834,0.020969,0.153259,0.014958,0.040249
2021-01-08,0.112842,0.116859,0.027327,0.049076,0.020526,0.019159,0.020268,0.025479,0.039611,0.055329,0.052605,0.201160,0.019401,0.009643,0.020473,0.155109,0.014709,0.040425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,0.076463,0.159440,0.037264,0.045512,0.004541,0.013054,0.020061,0.031890,0.061913,0.040717,0.076807,0.202649,0.009213,0.009580,0.072845,0.087334,0.011805,0.038913
2025-12-26,0.076823,0.159956,0.036842,0.045605,0.004615,0.013115,0.019986,0.032141,0.062224,0.040829,0.077249,0.201822,0.009401,0.009472,0.071064,0.088248,0.011795,0.038811
2025-12-29,0.077078,0.160756,0.036896,0.045011,0.004495,0.013045,0.019926,0.032211,0.062283,0.040677,0.077331,0.201383,0.009315,0.009351,0.070659,0.089181,0.011652,0.038750


In [22]:
log_ret.set_index('Date',inplace=True)
log_ret_stk = log_ret[stk_list].copy()


In [23]:
df_ret_w = log_ret_stk*weights
df_ret_w['industry weighted average return'] = df_ret_w.sum(axis=1)

In [24]:
mcd_ret = log_ret['MCD'] #还是要用原来的return算
ind_ret = df_ret_w['industry weighted average return']

df_reg = pd.concat([mcd_ret,ind_ret],axis=1)
df_reg.columns=['mcd','industry']

X = sm.add_constant(df_reg['industry'])
y = df_reg['mcd']

model = sm.OLS(y,X).fit()

alpha = model.params['const']
beta = model.params['industry']
r2 = model.rsquared


print("alpha:", alpha)
print("beta:", beta)
print("R-squared:", r2)


alpha: 0.00017461135006658588
beta: 0.4229947240888764
R-squared: 0.27737602837414055
